In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Nov 23 11:39:44 2020

@author: berar
"""
import numpy as np


GRID_SIZE = 4
TERMINAL_STATES = [0, GRID_SIZE*GRID_SIZE-1]
states = np.arange(GRID_SIZE*GRID_SIZE)
actions = ['UP', 'RIGHT', 'DOWN', 'LEFT']
discount = 1.

def next_state(grid_size, state, action):
    i,j = np.unravel_index(state, (grid_size, grid_size))
    if action == 'UP':
        i = np.maximum(0,i-1)
    elif action == 'DOWN':
        i = np.minimum(i+1,grid_size-1)
    elif action == 'RIGHT':
        j = np.minimum(j+1,grid_size-1)
    elif action == 'LEFT':
        j = np.maximum(0,j-1)    
    new_state = np.ravel_multi_index((i,j), (grid_size, grid_size))
    return new_state
    
def is_done(state, terminal_states):
    return state in terminal_states


# The unifom policy            
uniform_policy = {s : { a : 1/len(actions) for a in actions } for s in states}

# Transition is coded as a dictionary of dictionary
P = {}
for s in range(len(states)):
    P[s] = {a : () for a in actions}
    if s in TERMINAL_STATES:
        # if terminal state, stay where you are
        # instead of next_state
        reward = 0.
        for action in actions:
            P[s][action] = (s, reward, True)
    else:
        # transition
        reward = -1.
        for action in actions:
            next_s = next_state(GRID_SIZE, s, action)
            P[s][action] = (next_s,reward,is_done(next_s, TERMINAL_STATES))
 


 


Uni = {0: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 1: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 2: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 3: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 4: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 5: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 6: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 7: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 8: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 9: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 10: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 11: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 12: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 13: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 14: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}, 15: {'UP': 0.25, 'RIGHT': 0.25, 'DOWN': 0.25, 'LEFT': 0.25}}
{'UP': (0, 0.0, True), 'RIG

In [ ]:
theta = 1e-4  # Convergence threshold

# Initialize value function
V = np.zeros(len(states))

# Iterative Policy Evaluation
while True:
    delta = 0
    for s in states:
        if s in TERMINAL_STATES:
            continue
        v = V[s]
        new_v = 0
        for a in actions:
            prob = uniform_policy[s][a]  # 1/4
            next_s, reward, done = P[s][a]
            new_v += prob * (reward + discount * V[next_s])
        V[s] = new_v
        delta = max(delta, abs(v - V[s]))
    if delta < theta:
        break

# Pretty print the result as a grid
V_grid = V.reshape((GRID_SIZE, GRID_SIZE))
print("Value Function under Uniform Policy:")
print(np.round(V_grid, 2))

Value Function under Uniform Policy:
[[  0. -14. -20. -22.]
 [-14. -18. -20. -20.]
 [-20. -20. -18. -14.]
 [-22. -20. -14.   0.]]


In [14]:
def evaluate_policy(policy, P, states, terminal_states, actions, discount=1.0, theta=1e-4):
    V = np.zeros(len(states))
    while True:
        delta = 0
        for s in states:
            if s in terminal_states:
                continue
            v = V[s]
            new_v = 0
            for a in actions:
                prob = policy[s][a]
                next_s, reward, done = P[s][a]
                new_v += prob * (reward + discount * V[next_s])
            V[s] = new_v
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    return V

In [12]:
def policy_iteration(P, states, terminal_states, actions, discount=1.0, theta=1e-4):
    # Step 1: Initialize policy arbitrarily (uniform)
    policy = {s: {a: 1 / len(actions) for a in actions} for s in states}
    
    # Initial arbitrary deterministic policy for improvement step
    pi = {s: actions[0] for s in states if s not in terminal_states}
    
    while True:
        # Step 2: Policy Evaluation
        V = evaluate_policy(policy, P, states, terminal_states, actions, discount, theta)
        
        # Step 3: Policy Improvement
        policy_stable = True
        for s in states:
            if s in terminal_states:
                continue
            
            old_action = pi[s]
            
            # Evaluate all actions and pick the best one
            action_values = {}
            for a in actions:
                next_s, reward, done = P[s][a]
                action_values[a] = reward + discount * V[next_s]
            
            # Choose best action
            best_action = max(action_values, key=action_values.get)
            pi[s] = best_action
            
            # Update policy to deterministic (1.0 for best action)
            policy[s] = {a: 1.0 if a == best_action else 0.0 for a in actions}
            
            if best_action != old_action:
                policy_stable = False
        
        if policy_stable:
            break
    
    return V, pi


In [16]:
V_star, pi_star = policy_iteration(P, states, TERMINAL_STATES, actions)

# Format V_star into grid
print("Optimal Value Function:")
print(np.round(V_star.reshape(GRID_SIZE, GRID_SIZE), 2))

# Format policy into grid arrows
policy_grid = np.full((GRID_SIZE, GRID_SIZE), '', dtype='<U5')
arrow_map = {'UP': '↑', 'DOWN': '↓', 'LEFT': '←', 'RIGHT': '→'}

for s in states:
    if s in TERMINAL_STATES:
        i, j = np.unravel_index(s, (GRID_SIZE, GRID_SIZE))
        policy_grid[i, j] = 'T'
    else:
        i, j = np.unravel_index(s, (GRID_SIZE, GRID_SIZE))
        policy_grid[i, j] = arrow_map[pi_star[s]]

print("\nOptimal Policy:")
for row in policy_grid:
    print(' '.join(row))


Optimal Value Function:
[[ 0. -1. -2. -3.]
 [-1. -2. -3. -2.]
 [-2. -3. -2. -1.]
 [-3. -2. -1.  0.]]

Optimal Policy:
T ← ← ↓
↑ ↑ ↑ ↓
↑ ↑ → ↓
↑ → → T


In [17]:
def value_iteration(P, states, terminal_states, actions, discount=1.0, theta=1e-4):
    V = np.zeros(len(states))

    while True:
        delta = 0
        for s in states:
            if s in terminal_states:
                continue

            v = V[s]

            # Compute Q(s, a) for all actions
            q_values = []
            for a in actions:
                next_s, reward, done = P[s][a]
                q = reward + discount * V[next_s]
                q_values.append(q)

            V[s] = max(q_values)
            delta = max(delta, abs(v - V[s]))

        if delta < theta:
            break

    # Derive policy from final value function
    pi = {}
    for s in states:
        if s in terminal_states:
            continue

        action_values = {}
        for a in actions:
            next_s, reward, done = P[s][a]
            action_values[a] = reward + discount * V[next_s]

        # Best action for this state
        best_action = max(action_values, key=action_values.get)
        pi[s] = best_action

    return V, pi


In [18]:
V_star, pi_star = value_iteration(P, states, TERMINAL_STATES, actions)

# Format value function
print("Optimal Value Function (Value Iteration):")
print(np.round(V_star.reshape(GRID_SIZE, GRID_SIZE), 2))

# Visualize policy
policy_grid = np.full((GRID_SIZE, GRID_SIZE), '', dtype='<U5')
arrow_map = {'UP': '↑', 'DOWN': '↓', 'LEFT': '←', 'RIGHT': '→'}

for s in states:
    i, j = np.unravel_index(s, (GRID_SIZE, GRID_SIZE))
    if s in TERMINAL_STATES:
        policy_grid[i, j] = 'T'
    else:
        policy_grid[i, j] = arrow_map[pi_star[s]]

print("\nOptimal Policy (Value Iteration):")
for row in policy_grid:
    print(' '.join(row))


Optimal Value Function (Value Iteration):
[[ 0. -1. -2. -3.]
 [-1. -2. -3. -2.]
 [-2. -3. -2. -1.]
 [-3. -2. -1.  0.]]

Optimal Policy (Value Iteration):
T ← ← ↓
↑ ↑ ↑ ↓
↑ ↑ → ↓
↑ → → T


Now you have to apply the value iteration:

initilalize V
Repeat
    ∆ ← 0
    For each s ∈ S :
        v ← V (s)
        V (s) ← max_a Sum_(s′,r) p(s′, r |s, a)[r + γV (s′)]
        ∆ ← max(∆, |v − V (s)|)
until ∆ < θ (a small positive number)
Output a deterministic policy
π(s) = arg max_a Sum_(s′,r) p(s′, r |s, a)[r + γV (s′)]

input: π
v = 0|S|
Repeat
   ∆ ← 0
   For each s ∈ S :
        v ← V (s)
        V (s) ← Sum_a(π(a|s)) Sum_(s′,r) p(s′, r |s, a)[r + γV (s′)]
        ∆ ← max(∆, |v − V (s)|)
until ∆ < θ (a small positive number)
Output V ≈ vπ

Where v is the value, s are the states, a are actions, p(s′, r |s, a) is the enviromental model and r is the reward. Use γ = 1